# BMW Worldwide Sales Performance Analysis

**Author:** Siddharth Shenoy  
**Program:** MS Business Analytics & Information Management, Purdue University

This notebook analyzes BMW worldwide sales data from 2010–2024 using data cleaning, exploratory data analysis, statistical hypothesis testing, correlation analysis, and time-series analysis.


In [ ]:
# 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


In [ ]:
# 2. Load Dataset

# Recommended repository structure:
#
# BMW-Model-Sales-Performance-Analysis/
# ├── data/
# │   └── BMW.csv
# ├── notebooks/
# │   └── bmw_sales_analysis.ipynb
# └── README.md

DATA_PATH = "../data/BMW.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()


In [ ]:
# 3. Initial Data Inspection

print("Dataset Information:")
df.info()

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())


In [ ]:
# 4. Check Missing Values

missing_values = df.isnull().sum().sort_values(ascending=False)

print("Missing Values by Column:")
display(missing_values)


In [ ]:
# 5. Handle Missing Values

numeric_columns = df.select_dtypes(include=np.number).columns

df[numeric_columns] = df[numeric_columns].fillna(
    df[numeric_columns].mean()
)

df.dropna(inplace=True)

print("Remaining missing values:")
display(df.isnull().sum())

print("\nDataset shape after handling missing values:")
print(df.shape)


In [ ]:
# 6. Check and Remove Duplicates

duplicate_count = df.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

df = df.drop_duplicates().copy()

print("Dataset shape after duplicate removal:", df.shape)


In [ ]:
# 7. Remove Outliers Using IQR

def remove_iqr_outliers(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = dataframe[
        (dataframe[column] < lower_bound) |
        (dataframe[column] > upper_bound)
    ]

    print(f"{column}")
    print(f"Lower Bound: {lower_bound:,.2f}")
    print(f"Upper Bound: {upper_bound:,.2f}")
    print(f"Outliers Identified: {len(outliers):,}")

    cleaned_dataframe = dataframe[
        (dataframe[column] >= lower_bound) &
        (dataframe[column] <= upper_bound)
    ].copy()

    return cleaned_dataframe

df = remove_iqr_outliers(df, "Price_USD")
df = remove_iqr_outliers(df, "Sales_Volume")

print("\nFinal dataset shape after outlier removal:")
print(df.shape)


In [ ]:
# 8. Descriptive Statistics

print("Descriptive Statistics")

display(
    df.drop(
        columns=["Year"],
        errors="ignore"
    ).describe()
)


In [ ]:
# 9. Sales Volume by BMW Model

plt.figure(figsize=(16, 8))

sns.boxplot(
    data=df,
    x="Model",
    y="Sales_Volume"
)

plt.title("BMW Sales Volume by Model")
plt.xlabel("BMW Model")
plt.ylabel("Sales Volume")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 10. Average Sales by Model

model_sales = (
    df.groupby("Model")["Sales_Volume"]
    .mean()
    .sort_values(ascending=False)
)

display(model_sales)

plt.figure(figsize=(12, 7))

sns.barplot(
    x=model_sales.values,
    y=model_sales.index
)

plt.title("Average Sales Volume by BMW Model")
plt.xlabel("Average Sales Volume")
plt.ylabel("Model")
plt.tight_layout()
plt.show()


In [ ]:
# 11. Fuel Type vs Transmission

plt.figure(figsize=(12, 7))

ax = sns.countplot(
    data=df,
    x="Fuel_Type",
    hue="Transmission"
)

plt.title("Fuel Type vs Transmission")
plt.xlabel("Fuel Type")
plt.ylabel("Vehicle Count")

for container in ax.containers:
    ax.bar_label(container, fontsize=9)

plt.legend(
    title="Transmission",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


In [ ]:
# 12. Sales Records by Region

regional_counts = df["Region"].value_counts()

display(regional_counts)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=regional_counts.values,
    y=regional_counts.index
)

plt.title("BMW Sales Records by Region")
plt.xlabel("Number of Records")
plt.ylabel("Region")
plt.tight_layout()
plt.show()


In [ ]:
# 13. Price Distribution by Region

plt.figure(figsize=(14, 7))

sns.boxplot(
    data=df,
    x="Region",
    y="Price_USD"
)

plt.title("BMW Vehicle Price Distribution by Region")
plt.xlabel("Region")
plt.ylabel("Price (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 14. Average Vehicle Price by Region

regional_price = (
    df.groupby("Region")["Price_USD"]
    .mean()
    .sort_values(ascending=False)
)

display(regional_price)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=regional_price.values,
    y=regional_price.index
)

plt.title("Average BMW Vehicle Price by Region")
plt.xlabel("Average Price (USD)")
plt.ylabel("Region")
plt.tight_layout()
plt.show()


In [ ]:
# 15. Average Vehicle Price by Fuel Type

fuel_price = (
    df.groupby("Fuel_Type")["Price_USD"]
    .mean()
    .sort_values(ascending=False)
)

display(fuel_price)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=fuel_price.index,
    y=fuel_price.values
)

plt.title("Average BMW Vehicle Price by Fuel Type")
plt.xlabel("Fuel Type")
plt.ylabel("Average Price (USD)")
plt.tight_layout()
plt.show()


In [ ]:
# 16. Correlation Analysis

continuous_columns = [
    "Engine_Size_L",
    "Mileage_KM",
    "Price_USD",
    "Sales_Volume"
]

correlation_matrix = df[continuous_columns].corr()

display(correlation_matrix)

plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# 17. ANOVA: Vehicle Price by Fuel Type

print("H0: Average vehicle price is equal across fuel types.")
print("H1: At least one fuel type has a different average vehicle price.")

fuel_groups = [
    group["Price_USD"].values
    for _, group in df.groupby("Fuel_Type")
]

anova_stat_fuel, anova_p_fuel = stats.f_oneway(*fuel_groups)

print(f"\nF-statistic: {anova_stat_fuel:.4f}")
print(f"P-value: {anova_p_fuel:.6f}")

alpha = 0.05

if anova_p_fuel < alpha:
    print(
        "\nConclusion: Reject the null hypothesis. "
        "There is a statistically significant difference "
        "in average vehicle price among fuel types."
    )
else:
    print(
        "\nConclusion: Fail to reject the null hypothesis. "
        "No statistically significant difference was detected."
    )


In [ ]:
# 18. ANOVA: Vehicle Price by Region

print("H0: Average vehicle price is equal across regions.")
print("H1: At least one region has a different average vehicle price.")

region_groups = [
    group["Price_USD"].values
    for _, group in df.groupby("Region")
]

anova_stat_region, anova_p_region = stats.f_oneway(*region_groups)

print(f"\nF-statistic: {anova_stat_region:.4f}")
print(f"P-value: {anova_p_region:.6f}")

if anova_p_region < alpha:
    print(
        "\nConclusion: Reject the null hypothesis. "
        "At least one region has a significantly different "
        "average vehicle price."
    )
else:
    print(
        "\nConclusion: Fail to reject the null hypothesis. "
        "No significant regional price differences were detected."
    )


In [ ]:
# 19. Chi-Square Test: Fuel Type vs Transmission

print("H0: Fuel type and transmission are independent.")
print("H1: Fuel type and transmission are associated.")

contingency_table = pd.crosstab(
    df["Fuel_Type"],
    df["Transmission"]
)

display(contingency_table)

chi2, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")

if p_value < alpha:
    print(
        "\nConclusion: Reject the null hypothesis. "
        "Fuel type and transmission are statistically associated."
    )
else:
    print(
        "\nConclusion: Fail to reject the null hypothesis. "
        "No significant association was detected."
    )


In [ ]:
# 20. Sales Trend Over Time

annual_sales = (
    df.groupby("Year")["Sales_Volume"]
    .sum()
    .reset_index()
)

display(annual_sales)

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=annual_sales,
    x="Year",
    y="Sales_Volume",
    marker="o"
)

plt.title("BMW Sales Volume Over Time")
plt.xlabel("Year")
plt.ylabel("Total Sales Volume")
plt.tight_layout()
plt.show()


In [ ]:
# 21. Average Price Over Time

annual_price = (
    df.groupby("Year")["Price_USD"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=annual_price,
    x="Year",
    y="Price_USD",
    marker="o"
)

plt.title("Average BMW Vehicle Price Over Time")
plt.xlabel("Year")
plt.ylabel("Average Price (USD)")
plt.tight_layout()
plt.show()


In [ ]:
# 22. Top Models by Total Sales Volume

top_models = (
    df.groupby("Model")["Sales_Volume"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

display(top_models)

plt.figure(figsize=(12, 7))

sns.barplot(
    x=top_models.values,
    y=top_models.index
)

plt.title("Top 10 BMW Models by Total Sales Volume")
plt.xlabel("Total Sales Volume")
plt.ylabel("BMW Model")
plt.tight_layout()
plt.show()


In [ ]:
# 23. Business Insights Summary

print("BMW SALES PERFORMANCE ANALYSIS")
print("=" * 50)

print("\nKey Insights:")
print("1. BMW models demonstrate different levels of sales performance and variability.")
print("2. Vehicle pricing differs across geographic regions.")
print("3. Fuel type is associated with differences in average vehicle pricing.")
print("4. Fuel type and transmission configuration show a statistical relationship.")
print("5. Historical sales patterns provide information that can support forecasting and inventory planning.")

print("\nBusiness Recommendations:")
print("- Prioritize inventory and marketing for consistently high-performing BMW models.")
print("- Consider regional pricing strategies based on observed differences in vehicle price.")
print("- Monitor fuel-type preferences as electric and alternative powertrain adoption evolves.")
print("- Use historical sales trends to support production, inventory, and marketing decisions.")
print("- Extend the analysis with predictive modeling and sales forecasting.")
